In [ ]:
"""
Splitweave Symbolic Language — Test Notebook
=============================================
Tests all symbol categories: layout, transforms, deformations, signals,
and composed patterns evaluated via the recursive grid parser.
"""
import sys, os
import sympy as sp
import torch as th
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ensure splitweave / geolipi are importable
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import splitweave.symbolic as sws
import geolipi.symbolic as gls
from geolipi.torch_compute import Sketcher
from splitweave.torch_compute import grid_eval, rec_grid_eval

TILE_DIR = "/users/aganesh8/data/aganesh8/data/patterns/generative_tiles/food"
DEVICE = "cuda" if th.cuda.is_available() else "cpu"
RESOLUTION = 256

sketcher = Sketcher(device=DEVICE, resolution=RESOLUTION, n_dims=2)
print(f"Sketcher ready  |  device={DEVICE}  |  resolution={RESOLUTION}")

In [ ]:
# ── Visualization helpers (from splitweave.utils) ───────────────

from splitweave.utils.visualization import (
    grid_to_rgb,
    ids_to_rgb,
    signal_to_rgb,
    color_cells,
    show,
    PALETTE,
)

R = RESOLUTION

print("Visualization helpers ready")

## 1. Grid Instantiation

In [ ]:
# CartesianGrid and PolarGrid
cart = sws.CartesianGrid()
polar = sws.PolarGrid()

g_cart, _ = grid_eval(cart, sketcher)
g_polar, _ = grid_eval(polar, sketcher)

show([grid_to_rgb(g_cart), grid_to_rgb(g_polar)],
     ['CartesianGrid', 'PolarGrid'])

## 2. Rectangular Partitions

In [ ]:
# Rectangular partitions — grid coordinates & colored cell IDs
rect_exprs = {
    'RectRepeat':        sws.RectRepeat(sws.CartesianGrid(), (0.325, 0.25)),
    'RectShiftedX':      sws.RectRepeatShiftedX(sws.CartesianGrid(), (0.325, 0.25), (0.15,)),
    'RectShiftedY':      sws.RectRepeatShiftedY(sws.CartesianGrid(), (0.325, 0.25), (0.15,)),
    'RectInner':         sws.RectRepeatInner(sws.CartesianGrid(), (0.325, 0.25)),
    'RectFitting':       sws.RectRepeatFitting(sws.CartesianGrid(), (0.325, 0.25)),
}

imgs, titles = [], []
for name, expr in rect_exprs.items():
    g, gid = grid_eval(expr, sketcher)
    imgs.append(grid_to_rgb(g))
    titles.append(name + ' (grid)')
show(imgs, titles)

# Show cell IDs colored
imgs2, titles2 = [], []
for name, expr in rect_exprs.items():
    g, gid = grid_eval(expr.tensor(), sketcher)
    n = gid.sum(-1, keepdim=True)
    if gid is not None:
        imgs2.append(color_cells(n, PALETTE))
        titles2.append(name + ' (cells)')
show(imgs2, titles2)

## 3. Triangular & Diamond Partitions

In [ ]:
tri_dia_exprs = {
    'Triangular':  sws.TriangularRepeat(sws.CartesianGrid(), (0.25,)),
    'Diamond':     sws.DiamondRepeat(sws.CartesianGrid(), (0.25,)),
}

imgs, titles = [], []
for name, expr in tri_dia_exprs.items():
    g, gid = grid_eval(expr, sketcher)
    imgs.append(grid_to_rgb(g))
    titles.append(name + ' (grid)')
    if gid is not None:
        imgs.append(color_cells(gid, PALETTE))
        titles.append(name + ' (cells)')
show(imgs, titles)

## 4. Hexagonal Partitions

In [ ]:
hex_exprs = {
    'HexRepeat':   sws.HexRepeat(sws.CartesianGrid(), (0.25,)),
    'HexRepeatX':  sws.HexRepeatX(sws.CartesianGrid(), (0.25,)),
    'HexRepeatY':  sws.HexRepeatY(sws.CartesianGrid(), (0.25,)),
}

imgs, titles = [], []
for name, expr in hex_exprs.items():
    g, gid = grid_eval(expr, sketcher)
    imgs.append(grid_to_rgb(g))
    titles.append(name + ' (grid)')
    if gid is not None:
        imgs.append(color_cells(gid, PALETTE))
        titles.append(name + ' (cells)')
show(imgs, titles)

## 5. Radial / Polar Partitions

In [ ]:
# Radial partitions operate on a polar grid
polar_base = sws.CartToPolar(sws.CartesianGrid())

radial_exprs = {
    'RadialAngular':     sws.RadialRepeatAngular(polar_base, (0.25,)),
    'RadialCentered':    sws.RadialRepeatCentered(polar_base, (0.3, 0.3)),
    'RadialBricked':     sws.RadialRepeatBricked(polar_base, (0.25, 0.25), (0.1,)),
    'RadialFixedArc':    sws.RadialRepeatFixedArc(polar_base, (0.25, 0.4), (0.1,)),
}

imgs, titles = [], []
for name, expr in radial_exprs.items():
    g, gid = grid_eval(expr, sketcher)
    imgs.append(grid_to_rgb(g))
    titles.append(name + ' (grid)')
    if gid is not None:
        imgs.append(color_cells(gid, PALETTE))
        titles.append(name + ' (cells)')
show(imgs, titles)

## 6. Voronoi & Irregular Partitions

In [ ]:
# Voronoi / Irregular partitions -- no explicit base_grid needed;
# the evaluator injects base coords automatically for IrregularRectRepeat.
voronoi_exprs = {
    'Voronoi':         sws.VoronoiRepeat(sws.CartesianGrid(), (0.25, 0.27), (0.3,)),
    'IrregularRect':   sws.IrregularRectRepeat(sws.CartesianGrid(), (0.3, 0.3), (0.92,)),
    'Delaunay':        sws.DelaunayRepeat(sws.CartesianGrid(), (5.0, 5.0)),
}

imgs, titles = [], []
for name, expr in voronoi_exprs.items():
    try:
        g, gid = grid_eval(expr, sketcher)
        imgs.append(grid_to_rgb(g))
        n = gid.sum(-1, keepdim=True)
        titles.append(name + ' (grid)')
        if gid is not None:
            imgs.append(color_cells(n, PALETTE))
            titles.append(name + ' (cells)')
    except Exception as e:
        print(f"{name}: {e}")
if imgs:
    show(imgs, titles)

## 7. Grid Transforms

In [ ]:
# Apply transforms on a RectRepeat base
base = sws.CartesianGrid()

transform_exprs = {
    'Original':       base,
    'Translate':      sws.CartTranslate(base, (0.2, 0.1,)).tensor(),
    'Scale(1.5)':     sws.CartScale(base, (1.5,1.2)).tensor(),
    'Rotate(pi/6)':   sws.CartRotate(base, (0.5236,)).tensor(),
}

imgs, titles = [], []
for name, expr in transform_exprs.items():
    in_expr = sws.RectRepeat(expr, (0.35, 0.20))
    g, gid = grid_eval(in_expr, sketcher)
    if gid is not None:
        imgs.append(color_cells(gid, PALETTE))
    else:
        imgs.append(grid_to_rgb(g))
    titles.append(name)
show(imgs, titles)

## 8. Scalar Noise Fields

In [ ]:
# PerlinNoise and ValueNoise (Scalar2D)
perlin = sws.PerlinNoise((8,))
value  = sws.ValueNoise((8,))

g_perlin, _ = grid_eval(perlin, sketcher)
g_value, _  = grid_eval(value, sketcher)

show([signal_to_rgb(g_perlin), signal_to_rgb(g_value)],
     ['PerlinNoise(8)', 'ValueNoise(8)'])

# Multi-octave via expression arithmetic
multi = sws.PerlinNoise((4,)).tensor() + sws.PerlinNoise((8,)).tensor() + sws.PerlinNoise((16,)).tensor()
g_multi, _ = grid_eval(multi, sketcher)
show([signal_to_rgb(g_multi)], ['Perlin 4+8+16'])

## 9. Signal-Driven Transforms

In [ ]:
# Use Perlin noise to drive grid warping via signal transforms
# Build expressions with tuples, then tensorize at execution time.
base = sws.CartesianGrid()
noise = sws.PerlinNoise((4,)) + sws.PerlinNoise((8,))
translate_noise = noise * gls.Param((0.1,))
scale_noise = gls.Param((1.0,)) + noise * gls.Param((0.1,))

signal_tx_exprs = {
    'TranslateWtSignal': sws.TranslateWtSignal(base, translate_noise),
    'ScaleWtSignal':     sws.ScaleWtSignal(base, scale_noise),
}

imgs, titles = [], []

for name, expr in signal_tx_exprs.items():
    # Tensorize the full expression on the current device before execution
    in_expr = sws.RectRepeat(expr, (0.35, 0.15)).tensor(device=DEVICE)
    g, gid = grid_eval(in_expr, sketcher)
    if gid is not None:
        imgs.append(color_cells(gid, PALETTE))
    else:
        imgs.append(grid_to_rgb(g))
    titles.append(name)
    #     print(f"{name}: {e}")
show(imgs, titles)

## 10. Deformations

In [ ]:
# Apply each deformation to a RectRepeat grid
base = sws.CartesianGrid()
center = th.tensor([0.0, 0.0]).cuda()

deform_exprs = {
    'Radial(linear)':  sws.RadialDeform(base, center, sp.Symbol('linear'), (0.5,)),
    'Radial(siny)':    sws.RadialDeform(base, center, sp.Symbol('siny'), (0.5,), (6.0,), (0.0,)),
    'Radial(sigmoid)': sws.RadialDeform(base, center, sp.Symbol('sigmoid'), (0.5,),
                                         (0.0,), (0.0,), (-20.0,), (0.35,)),
    'Perlin(xy)':      sws.PerlinDeform(base, (4, 8, 16), (42,),
                                         sp.Symbol('xy'), (0.15,)),
    'Decay(x)':        sws.DecayDeform(base, (1.0, 0.5),
                                        sp.Symbol('x'), (0.6,)),
    'Strip(xy)':       sws.StripDeform(base, (0.785,), sp.Symbol('xy'), (4.0,), (0.0,), (0.08,)),
    'Swirl(linear)':   sws.SwirlDeform(base, center, sp.Symbol('linear'), (2.0,)),
}

imgs, titles = [], []
for name, expr in deform_exprs.items():
    in_expr = sws.RectRepeat(expr, (0.35, 0.20,))
    g, gid = grid_eval(in_expr.tensor(), sketcher)
    if gid is not None:
        imgs.append(color_cells(gid, PALETTE))
    else:
        imgs.append(grid_to_rgb(g))
    titles.append(name)

show(imgs[:4], titles[:4])
show(imgs[4:], titles[4:])

## 11. Continuous Signals

In [ ]:
# Evaluate each signal type as a scalar field and visualize
center = (0, 0)

signal_exprs = {
    'Radial(linear)':   sws.RadialSignal(center, sp.Symbol('linear')),
    'Radial(siny)':     sws.RadialSignal(center, sp.Symbol('siny'), (6.0,), (0.0,)),
    'Radial(sigmoid)':  sws.RadialSignal(center, sp.Symbol('sigmoid'),
                                          (0.0,), (0.0,), (-20.0,), (0.35,)),
    'Perlin':           sws.PerlinSignal((4, 8, 16), (42,)),
    'Decay':            sws.DecaySignal((1.0, 0.5), sp.Symbol('xy')),
    'Strip':            sws.StripSignal((0.785,), (4.0,), (0.0,)),
    'Swirl(siny)':      sws.SwirlSignal(center, sp.Symbol('siny'), (6.0,), (0.0,)),
}

imgs, titles = [], []
for name, expr in signal_exprs.items():
    s, _ = grid_eval(expr.tensor(), sketcher)
    imgs.append(signal_to_rgb(s))
    titles.append(name)

show(imgs[:4], titles[:4])
show(imgs[4:], titles[4:])

## 12. Colored Patterns (grid_ids to RGBA)

In [ ]:
# Produce actual colored RGBA pattern images.
# Method: grid_ids % n_colors → palette index → per-pixel RGBA canvas.

from geolipi.torch_compute.color_functions import source_over_seq

def make_colored_canvas(grid_ids, palette, resolution, col_idx=-1):
    """Build (R*R, 4) RGBA canvas by coloring cells from a palette (N, 3)."""
    ids = grid_ids[..., col_idx].long()
    n = palette.shape[0]
    rgb = palette[ids % n].to(grid_ids.device)
    alpha = th.ones(rgb.shape[0], 1, device=rgb.device)
    return th.cat([rgb, alpha], dim=-1)

# Build several colored patterns
patterns = {
    'RectRepeat':     sws.RectRepeat(sws.CartesianGrid(), (0.2, 0.2)),
    'HexRepeat':      sws.HexRepeat(sws.CartesianGrid(), (0.2,)),
    'Brick (ShiftX)': sws.RectRepeatShiftedX(sws.CartesianGrid(), (0.2, 0.3), (0.5,)),
    'Diamond':        sws.DiamondRepeat(sws.CartesianGrid(), (0.2,)),
}

pal = PALETTE.to(DEVICE)
imgs, titles = [], []
for name, expr in patterns.items():
    g, gid = grid_eval(expr, sketcher)
    if gid is not None:
        canvas = make_colored_canvas(gid, pal, R)
        img = canvas.detach().cpu().reshape(R, R, 4).numpy()
        imgs.append(img[..., :3])
        titles.append(name)
show(imgs, titles)

## 13. Composed Multi-Stage Expressions

In [ ]:
# Compose: Deform → Partition, Transform → Deform → Partition, etc.
center = th.tensor([0.0, 0.0])

# 1) Swirl deform → then RectRepeat
expr_swirl_rect = sws.RectRepeat(
    sws.SwirlDeform(sws.CartesianGrid(), center, sp.Symbol('linear'), (1.5,)),
    (0.2, 0.2)
)

# 2) Scale → RadialDeform → HexRepeat
expr_scale_radial_hex = sws.HexRepeat(
    sws.RadialDeform(
        sws.CartScale(sws.CartesianGrid(), (1.3,)),
        center, sp.Symbol('siny'), (0.4,), (6.0,), (0.0,)
    ),
    (0.2,)
)

# 3) StripDeform → Rotate → RectRepeat (wavy brick)
expr_strip_rotate_rect = sws.RectRepeatShiftedX(
    sws.CartRotate(
        sws.StripDeform(sws.CartesianGrid(), (0.0,), sp.Symbol('xy'), (6.0,), (0.0,), (0.06,)),
        (0.3,)
    ),
    (0.2, 0.3,), (0.5,)
)

# 4) PerlinDeform → Diamond
expr_perlin_diamond = sws.DiamondRepeat(
    sws.PerlinDeform(sws.CartesianGrid(), th.tensor([4, 8]), (7,),
                      sp.Symbol('xy'), (0.12,)),
    (0.2,)
)

composed = {
    'Swirl->Rect':           expr_swirl_rect,
    'Scale->Radial->Hex':    expr_scale_radial_hex,
    'Strip->Rot->BrickX':    expr_strip_rotate_rect,
    'Perlin->Diamond':       expr_perlin_diamond,
}

pal = PALETTE.to(DEVICE)
imgs, titles = [], []
for name, expr in composed.items():
    g, gid = grid_eval(expr.tensor(), sketcher)
    if gid is not None:
        canvas = make_colored_canvas(gid, pal, R)
        imgs.append(canvas.detach().cpu().reshape(R, R, 4).numpy()[..., :3])
    else:
        imgs.append(grid_to_rgb(g))
    titles.append(name)
show(imgs, titles)

## 14. Edge Masks

In [ ]:
# Edge masks alongside their partition counterparts.
# Edge symbols use the same base class (PartitionGrid) so the same
# dispatch handles them, but the underlying function returns edge masks.

edge_pairs = {
    'Rect': (
        sws.RectRepeat(sws.CartesianGrid(), (0.25, 0.25)),
        sws.RectRepeatEdge(sws.CartesianGrid(), (0.25, 0.25)),
    ),
    'Hex': (
        sws.HexRepeat(sws.CartesianGrid(), (0.25,)),
        sws.HexRepeatEdge(sws.CartesianGrid(), (0.25,)),
    ),
}

imgs, titles = [], []
for name, (part_expr, edge_expr) in edge_pairs.items():
    # Partition
    g, gid = grid_eval(part_expr, sketcher)
    if gid is not None:
        imgs.append(color_cells(gid, PALETTE))
    else:
        imgs.append(grid_to_rgb(g))
    titles.append(f'{name} Partition')

    # Edge
    try:
        g_e, gid_e = grid_eval(edge_expr, sketcher)
        # Edge grids are typically scalar — show as signal
        imgs.append(signal_to_rgb(g_e))
        titles.append(f'{name} Edge')
    except Exception as e:
        print(f"{name} Edge: {e}")

show(imgs, titles)

## 15. Tile Placement (geolipi TileUV2D + splitweave grid)

In [ ]:
# Place a tile into grid cells using geolipi's TileUV2D + recursive_evaluate.
# Steps:
#   1. Evaluate splitweave grid expression → (grid, grid_ids)
#   2. Build a geolipi TileUV2D expression using the grid as coordinates
#   3. Evaluate with geolipi's recursive_evaluate

import geolipi.symbolic.primitives_2d as prim2d
from geolipi.torch_compute import recursive_evaluate

# Load first tile from TILE_DIR
tile_files = sorted(os.listdir(TILE_DIR))
if tile_files:
    tile_file = np.random.choice(tile_files)
    tile_path = os.path.join(TILE_DIR, str(tile_file))
    tile_pil = Image.open(tile_path).convert('RGBA').resize((512, 512))
    tile_np = np.array(tile_pil).astype(np.float32) / 255.0

    # PIL is HxWxC, need WxHxC for TileUV2D (transposed)
    tile_t = th.from_numpy(tile_np).to(DEVICE)
    
    

    # Evaluate a grid expression (symbolic -> tensor via grid_eval)
    expr = sws.RectRepeat(sws.CartesianGrid(), (0.2, 0.4))
    grid, grid_ids = grid_eval(expr, sketcher)

    # Use grid as UV coordinates for the tile
    tile_expr = gls.Scale2D(prim2d.TileUV2D(tile_t), (0.7, 0.7)).tensor()
    canvas = recursive_evaluate(tile_expr, sketcher, coords=grid)

    # Visualize
    print(canvas.shape)
    canvas_img = canvas.detach().cpu().reshape(R, R, 4).numpy()
    show([canvas_img[..., :3], color_cells(grid_ids, PALETTE)],
         [f'Tile: {tile_files[0]}', 'RectRepeat cells'])
else:
    print(f"No tile files found in {TILE_DIR}")